# Respiration: gyroscope evidence to reconsider lower-back nonwear

Start with **subject 900001 / T0**. The hypothesis is that respiration-like periodic
rotation in **at least one gyroscope axis** provides evidence of wear during an
interval labelled nonwear by the acceleration detector.

This notebook compares **30 s / 15 s hop** and **60 s / 30 s hop** (50% overlap).
It screens each gyro axis independently, then tests:

`respiration_refined_nonwear = original_nonwear AND NOT respiration_evidence`

Then the final rule reclassifies residual bouts shorter than
`MIN_NONWEAR_BOUT_SECONDS = 900` as wear. Exactly 15 minutes remains nonwear.
Changing only this cutoff requires rerunning the override/plot cells, not the FFTs.

A positive window reassigns **only its overlap with existing nonwear** to wear.
Overlapping windows are unioned before counting; a single positive window does
not relabel a whole long nonwear bout. The time support is the full positive
window, so boundaries have 30/60-second uncertainty. No new nonwear is created.

**This is an exploratory screen, not a validated respiration detector.** Periodic
movement or external vibration may look similar. The starting thresholds below
are editable hypotheses, not published or calibrated cutoffs. Absence of a peak
is not evidence of nonwear. Check known wear and known removals before interpreting
reassigned intervals as corrections.

This notebook writes separate experiment outputs. The same spectral screen is now
available as an optional production override, disabled by default. Detection uses
10 < BR < 30 brpm at all hours; daytime masking below only affects the figure.


The shared detector now uses **10 < BR < 30 brpm at all hours**. Rerun the analysis and override cells to refresh historical outputs; plotting masks only affect display. Production integration uses a Python port of the pinned upstream `ggir_2023_matched` baseline (two axes, SD 13 mg / range 50 mg). Production needs no R runtime or source downloads.

In [ ]:
from pathlib import Path
from dataclasses import asdict
import importlib
import json
import sys
import tomllib

PROJECT_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (p / "scripts" / "lower_back_respiration.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or its notebooks directory.")
for folder in (PROJECT_ROOT / "scripts", PROJECT_ROOT / "src"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import lower_back_respiration as rp
rp = importlib.reload(rp)

# Optional interactive window backend: put %matplotlib qt in a separate cell.


## Inspect saved results: raw ACC/gyro and nonwear

Run the imports above and the **two cells below** to inspect the saved cohort
results. The comparison, FFT and exploratory override cells farther down are not
needed. Nothing is recomputed or written to the saved results.

The plot shows **raw acceleration above raw gyroscope**, retaining every original
sample (100 Hz). Shading distinguishes **blue = final nonwear**, **green =
reassigned to wear by respiration**, and **purple = reassigned by the <15-minute
rule**. Gray hatching marks missing/time-invalid spans. Labels remain visible at
all hours. Acceleration is in g before autocalibration; gyro is in degrees/s.

Start with subject **900001 / T0**. Change `INSPECT_SUBJECT`/`INSPECT_VISIT` to inspect
another participant (keep leading zeroes). `INSPECTION_RUN = None` selects the
latest completed result containing that participant; set an explicit `Path` to pin a run.
The output root is located beside the configured assessment output folder and
can be changed below. New 60-minute production results are selected when available. Otherwise the
older 197-participant, 60-minute cohort results are used and clearly labelled
as historical. Each saved profile is printed above the plot; saved labels are
never silently treated as results from a different window duration.


In [ ]:
import inspect_lower_back_nonwear as ni
from dare_wearables.lower_back.nonwear.ggir2023 import NonwearOptions
ni = importlib.reload(ni)

INSPECT_SUBJECT = "900001"
INSPECT_VISIT = "T0"
INSPECTION_CONFIG = PROJECT_ROOT / "configs" / "lower_back_respiration.local.toml"
with INSPECTION_CONFIG.open("rb") as stream:
    inspection_config = tomllib.load(stream)["lower_back"]
inspection_profile = NonwearOptions.from_mapping(inspection_config.get("nonwear"))
assessment_output = Path(inspection_config["silver_root"]).expanduser()
if not assessment_output.is_absolute():
    assessment_output = INSPECTION_CONFIG.parent / assessment_output
INSPECTION_ROOT = assessment_output
HISTORICAL_INSPECTION_ROOT = assessment_output.parent / "lower_back_minimum_bout_assessment"
INSPECTION_RUN = None  # or Path("/absolute/path/to/duration_...")
SAVED_RUN = ni.find_saved_run(
    INSPECTION_ROOT, run=INSPECTION_RUN, subject=INSPECT_SUBJECT, visit=INSPECT_VISIT,
    fallback_roots=(HISTORICAL_INSPECTION_ROOT,),
)
inspection = ni.load_saved_results(SAVED_RUN, INSPECT_SUBJECT, INSPECT_VISIT)
print(f"Saved results: {SAVED_RUN}")
print(f"Saved baseline: {inspection['baseline_window_seconds']/60:g}-min windows / 15-min step; "
      f"end extension={inspection['end_of_recording_correction']}")
if (inspection["baseline_window_seconds"] != inspection_profile.window_seconds
        or inspection["end_of_recording_correction"] != inspection_profile.end_of_recording_correction):
    print("Historical profile: this participant has not yet been recomputed with the current production settings.")
print(f"{INSPECT_SUBJECT}/{INSPECT_VISIT} · {inspection['sample_rate_hz']:g} Hz · "
      f"minimum final nonwear {inspection['min_nonwear_bout_seconds']/60:g} min")
display(pd.DataFrame([
    {"category": name, "bouts": len(table),
     "hours": table.duration_seconds.sum() / 3600}
    for name, table in inspection["intervals"].items()
]))
# bout_index is the index within its category, used by the plotting cell below.
display(inspection["review_bouts"].head(40))


Choose a category and `FOCUS_BOUT_INDEX` from the table, or replace `RAW_START`
and `RAW_END` with recording-local timestamp strings. The default view is 20
minutes around the start of the first respiration reassignment. If that category
has no bouts, the view starts near the recording beginning.

Set **both `RAW_START = RAW_END = None`** to display the whole analysed recording,
up to seven days. A full-week raw plot can be slow; a short view is best for
examining respiratory waves. The first plot reads the OMX and may need substantial
RAM; subsequent views reuse the cached raw recording. All saved bout boundaries
are checked against the loaded samples, including the timestamp-recovery policy.


In [ ]:
%matplotlib qt

In [ ]:
FOCUS_CATEGORY = "respiration"  # "final_nonwear", "short_bouts", or "baseline"
FOCUS_BOUT_INDEX = 0
focus_bouts = inspection["intervals"][FOCUS_CATEGORY]
if focus_bouts.empty:
    focus_start = pd.Timestamp(inspection["source_summary"]["start"])
else:
    focus_start = focus_bouts.iloc[FOCUS_BOUT_INDEX].start

# RAW_START = focus_start - pd.Timedelta(minutes=5)  # or "2026-03-11 03:25:00"
# RAW_END = focus_start + pd.Timedelta(minutes=15)   # or "2026-03-11 04:00:00"
RAW_START = RAW_END = None  # full analysed recording; may be slow at raw 100 Hz

raw_key = (str(inspection["source"]), inspection["input_identity"],
           inspection["n_samples"], inspection["timestamp_policy"])
if globals().get("_nw_inspection_raw_cache", {}).get("key") != raw_key:
    _nw_inspection_raw_cache = {}  # release the previous participant before loading another
    print("Loading and checking raw ACC/gyro once; later plot views reuse these samples.")
    _nw_inspection_raw_cache = {"key": raw_key, "raw": ni.load_saved_raw(inspection)}
raw_nonwear_figure = ni.plot_saved_raw(
    _nw_inspection_raw_cache["raw"], inspection, start=RAW_START, end=RAW_END,
)
raw_nonwear_figure

In [ ]:
# Load the cohort summary belonging to the selected saved run.
df = pd.read_csv(SAVED_RUN / "cohort_summary.csv")
df.plot(x="subject", y="revised_nonwear_hours", kind="bar")


## Select the existing comparison and load 900001 once

Use the private configuration from the nonwear comparison notebook to locate its
output root. `COMPARISON_RUN = None` selects the latest **completed** run containing
900001/T0 and the selected baseline. To fix the input explicitly, set it to a saved
run directory. An older two-axis matched run works; the retired 150 mg variant is
ignored. This notebook does not launch a comparison or R process.

The default plotted baseline is GGIR matched, at least two axes. If available,
the current Python and matched three-axis results are also included in the summary.
Only the original analysed sample sequence is loaded, using the same reader and
seven-day sample cap. Gyro units are **degrees/second**; no temperature is involved.
A full recording still needs substantial memory with the existing OMX reader.

This exploratory section uses the historical window settings recorded in the
selected comparison. For the current 60-minute production labels, use the saved-results
inspection section above. Changing production configuration does not alter old comparisons.


In [ ]:
SUBJECT = "900001"
VISIT = "T0"
BASELINE_METHOD = "ggir_2023_matched"
MIN_NONWEAR_BOUT_SECONDS = 900.0  # final bouts shorter than 15 min become wear
# Alternatives: "current", "ggir_2023_matched_3axes" (if saved in your run).
CONFIG_PATH = PROJECT_ROOT / "configs" / "nonwear_comparison.local.toml"
COMPARISON_RUN = None  # or Path("/your/private/comparison/run_...")

with CONFIG_PATH.open("rb") as stream:
    comparison_settings = tomllib.load(stream)["comparison"]
OUTPUT_ROOT = Path(comparison_settings["output_root"]).expanduser()
if not OUTPUT_ROOT.is_absolute():
    OUTPUT_ROOT = CONFIG_PATH.parent / OUTPUT_ROOT
OUTPUT_ROOT = OUTPUT_ROOT.resolve()
RUN_DIR = rp.find_comparison_run(OUTPUT_ROOT, SUBJECT, VISIT,
                                 method=BASELINE_METHOD, run=COMPARISON_RUN)
print(f"Using completed comparison: {RUN_DIR}")


In [ ]:
gyro, FS, recording_info, baselines = rp.load_recording(RUN_DIR, SUBJECT, VISIT)
BASELINE = baselines[BASELINE_METHOD]
display(pd.DataFrame([{
    "subject": SUBJECT, "visit": VISIT, "sample_rate_hz": FS,
    "observed_hours": len(gyro) / FS / 3600,
    "start": gyro.index[0], "end": recording_info["end"],
    "available_baselines": ", ".join(baselines),
}]))


## Editable spectral screen

For each complete, gap-free window and each axis:

1. Compute a **linear-detrended Hann periodogram** of the unfiltered 100 Hz signal.
   There is no prior respiratory bandpass, resampling or zero-padding. Fourier-bin
   spacing is **2 cycles/min at 30 s** and **1 cycle/min at 60 s**; Hann tapering
   broadens peaks, so those numbers are not claimed rate precision.
2. Find the strongest peak in the exploratory **0.10–0.60 Hz (10–30 cycles/min)** band.
   Require a local maximum and all amplitude/concentration conditions below.
3. Call a window positive if **any one axis** passes all conditions.

The band power fraction uses 0.05–2 Hz as its reference. Peak concentration is
power within ±0.05 Hz of the peak (at least one Fourier bin), divided by all power
in the candidate band. Peak/background compares peak PSD with median PSD outside
that peak neighbourhood within the reference band; it is a descriptive ratio,
not a statistical significance test. Band RMS is the square root of integrated
candidate-band PSD, in degrees/second.

The low-amplitude floor matters: normalized fractions alone can accept tiny noise.
The optional `max_gyro_std_dps` rejects high-amplitude windows; it starts disabled
because its scale needs inspection in this subject. A broad reference band and
peak concentration help screen broad movement, but cannot prove respiration.


In [ ]:
WINDOW_SECONDS = [60]
OVERLAP = 0.50
SETTINGS = rp.RespirationSettings(
    band_hz=(10 / 60, 30 / 60),
    reference_band_hz=(0.05, 2.0),
    min_band_rms_dps=0.01,
    min_band_power_fraction=0.50,
    min_peak_concentration=0.50,
    min_peak_to_background_db=6.0,
    peak_half_width_hz=0.05,
    max_gyro_std_dps=None,  # optional movement guard; inspect before choosing a value
)
display(pd.Series(asdict(SETTINGS), name="Exploratory starting settings"))


## Analyze both window lengths

This cell performs the spectral analysis when **you** run it. It can take time
for seven days. FFTs run in batches, retaining only PSD frequencies through 2 Hz
for the plots. No window crosses a timestamp gap. A nonfinite axis is ignored;
another valid axis can still provide evidence. Short segments and incomplete
window tails provide no evidence, so existing labels remain unchanged there.

Changing only the baseline or display interval does not require rerunning this
cell. Changing a spectral threshold, window length or overlap does.


In [ ]:
results = {}
for seconds in WINDOW_SECONDS:
    print(f"Analyzing {SUBJECT}/{VISIT}: {seconds} s windows", flush=True)
    results[seconds] = rp.analyze_gyro(
        gyro, FS, window_seconds=seconds, overlap=OVERLAP,
        settings=SETTINGS, batch_size=128,
    )

display(pd.DataFrame([{
    "window_seconds": seconds, "hop_seconds": seconds * (1 - OVERLAP),
    "frequency_bin_cycles_per_min": 60 / seconds,
    "windows": len(result.windows),
    "valid_windows": result.windows.valid_axes.gt(0).sum(),
    "positive_windows": result.windows.respiration_present.sum(),
} for seconds, result in results.items()]))


In [ ]:
%matplotlib qt
# Plot the RR rate across the recording, with a moving average to smooth out the noise.
plt.figure(figsize=(12, 4))
plt.subplot(2, 1, 1)
plt.plot(results[60].windows.gx_peak_hz.rolling(5, min_periods=1).mean() * 60, label="RR Rate (5-window moving average)")
plt.plot(results[60].windows.gy_peak_hz.rolling(5, min_periods=1).mean() * 60, label="RR Rate (30-window moving average)")
plt.plot(results[60].windows.gz_peak_hz.rolling(5, min_periods=1).mean() * 60, label="RR Rate (30-window moving average)")
plt.title(f"{SUBJECT}/{VISIT} RR Rate")
plt.xlabel("Time")
plt.ylabel("RR Rate")
plt.legend()

## Compare nonwear before and after the proposed override

The table includes all available current/matched baselines, never the retired
GGIR default thresholds. Durations count **observed samples**, so gaps are not
counted as respiration, wear or nonwear. Positive overlapping windows are counted
once. Unassessed nonwear is shown separately. Respiration does not reassign it;
the final duration rule can still reassign residual bouts shorter than 15 minutes.

The green portion of each bar is **proposed reassignment**, not measured accuracy.
A larger reassignment is not necessarily better.


In [ ]:
overrides = {}
summary_rows = []
for seconds, result in results.items():
    for method, baseline in baselines.items():
        override = rp.apply_override(
            result, baseline, gyro.index,
            min_nonwear_bout_seconds=MIN_NONWEAR_BOUT_SECONDS,
        )
        overrides[(seconds, method)] = override
        summary_rows.append({"subject": SUBJECT, "visit": VISIT,
                             "baseline_method": method, **override["summary"]})
respiration_summary = pd.DataFrame(summary_rows)
display(respiration_summary)

effect_figure, effect_axis = plt.subplots(figsize=(12, 5))
respiration_summary.set_index(["baseline_method", "window_seconds"])[
    ["revised_nonwear_hours", "reassigned_to_wear_hours"]
].rename(columns={"revised_nonwear_hours": "Remaining nonwear",
                  "reassigned_to_wear_hours": "Proposed reassignment to wear"}).plot.bar(
    stacked=True, color=["#315fc4", "#19715b"], ax=effect_axis,
)
effect_axis.set(ylabel="Observed hours", xlabel="Baseline / window duration (s)",
                title=f"{SUBJECT} / {VISIT} — each bar sums to original detected nonwear")
effect_figure.tight_layout()


## Spectral maps and nonwear intervals

Start with the full analysed recording (`PLOT_START = PLOT_END = None`), then
select an interval using recording-local timestamps. Change
`DISPLAY_WINDOW_SECONDS` to compare 30 and 60 s, or `BASELINE_METHOD` above and
rerun this cell to compare another available baseline.

There is no raw-gyro panel in this overview. The three spectral maps show
**0–50 brpm** (`BRPM_YLIM`) throughout the full day. Cyan positive-axis markers
retain the strict **10 < BR < 30 brpm** filter (`BRPM_LIMITS`).

`MASKED_DAYTIME = ("09:00", "21:00")` hides **only the cyan BR dots and the
Respiration-like evidence row** during those hours. PSD, Original nonwear,
Respiration-based reassignments, short-bout reassignments and remaining nonwear stay visible throughout the day.
Set it to `None` to show dots/evidence at all hours. The seven-day time axis
is preserved; recording gaps stay blank. Spectral colors use all displayed PSD.

These are **display filters only**. Detector thresholds, underlying evidence,
before/after nonwear masks, and duration summaries remain unchanged. The
nonwear classifications are not recomputed using the display filters.


In [ ]:
DISPLAY_WINDOW_SECONDS = 60
PLOT_START = None  # or a recording-local timestamp string
PLOT_END = None
MASKED_DAYTIME = ("09:00", "21:00")  # hide BR dots/evidence only; None disables
BRPM_LIMITS = (10.0, 30.0)  # strict bounds for cyan markers only
BRPM_YLIM = (0.0, 50.0)  # spectral background and y-axis range

# Reload plotting changes while retaining the already-computed analysis.
rp = importlib.reload(rp)
result = results[DISPLAY_WINDOW_SECONDS]
BASELINE = baselines[BASELINE_METHOD]
override = overrides[(DISPLAY_WINDOW_SECONDS, BASELINE_METHOD)]
overview_figure = rp.plot_overview(gyro, result, BASELINE, override,
                                   start=PLOT_START, end=PLOT_END,
                                   masked_daytime=MASKED_DAYTIME, brpm_limits=BRPM_LIMITS,
                                   brpm_ylim=BRPM_YLIM)
overview_figure


## Inspect candidate windows inside detected nonwear

This table ranks windows overlapping the selected baseline nonwear. It shows
all axes so a result cannot hide behind selecting only the strongest axis.
`baseline_nonwear_fraction` is the fraction of observed window samples that were
nonwear. These are candidate review intervals, not confirmed false nonwear.


In [ ]:
window_table = result.windows.copy()
window_table["baseline_nonwear_fraction"] = rp.nonwear_fraction(
    window_table, BASELINE, result.n_samples,
)
nonwear_windows = window_table.loc[window_table.baseline_nonwear_fraction.gt(0)]
ranked_windows = nonwear_windows.sort_values(["positive_axes", "start"], ascending=[False, True])
review_columns = ["start", "end", "baseline_nonwear_fraction", "positive_axes"]
for axis in rp.AXES:
    review_columns.extend(f"{axis}_{feature}" for feature in (
        "positive", "peak_hz", "band_rms_dps", "band_power_fraction",
        "peak_concentration", "peak_to_background_db",
    ))
display(ranked_windows[review_columns].head(20))


## Raw 100 Hz gyro and the exact spectrum for one window

Set `INSPECT_AT` to a recording-local timestamp, or leave it `None` to inspect the
first positive window overlapping nonwear (falling back to nonwear, then any
window). The closest window center is used for a supplied timestamp; the selected
start/end are printed so a timestamp in a gap is apparent.

All original samples appear on the left, with only their mean removed for display.
The right-hand PSD is the same detrended/tapered PSD used for the decision. Green
shading marks the candidate band and the red line marks its strongest peak.
Look for repeated breathing-like cycles rather than one large transient.


In [ ]:
INSPECT_AT = None
if INSPECT_AT is None:
    candidates = ranked_windows.loc[ranked_windows.respiration_present]
    if candidates.empty:
        candidates = ranked_windows if not ranked_windows.empty else window_table
    selected_window = int(candidates.index[0])
else:
    target = pd.Timestamp(INSPECT_AT)
    selected_window = int((result.windows.center - target).abs().idxmin())
display(window_table.loc[[selected_window], review_columns])
window_figure = rp.plot_window(gyro, result, selected_window)
window_figure


## See where the amplitude and relative-power thresholds lie

Each point is one window/axis. Green points passed the **complete** screen;
gray points did not. Dashed lines show two of the thresholds; concentration,
peak/background and local-maximum checks still apply. This is a threshold
inspection plot, not evidence that green points are true respiration.


In [ ]:
quality_figure, quality_axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for axis, name in zip(quality_axes, rp.AXES):
    valid = result.windows[f"{name}_valid"]
    data = result.windows.loc[valid]
    colors = np.where(data[f"{name}_positive"], "#19715b", "#999999")
    axis.scatter(np.maximum(data[f"{name}_band_rms_dps"], 1e-8),
                 data[f"{name}_band_power_fraction"], c=colors, s=4, alpha=.35)
    axis.axvline(result.settings.min_band_rms_dps, color="#c0392b", ls="--")
    axis.axhline(result.settings.min_band_power_fraction, color="#c0392b", ls="--")
    axis.set(xscale="log", xlabel="Band RMS (°/s)", title=name, ylim=(-.02, 1.02))
quality_axes[0].set_ylabel("Band / reference power")
quality_figure.suptitle(f"{DISPLAY_WINDOW_SECONDS} s windows — inspect threshold sensitivity")
quality_figure.tight_layout()


## Optional: check independently known wear and removals

Add intervals you can identify from removal logs or other independent evidence.
Use `label="wear"` or `label="nonwear"`. Avoid defining labels from the gyro peak
itself, which would make the assessment circular. Include confirmed removals to
check whether bed vibration or handling produces false respiration evidence.

Only complete windows fully contained in one labelled interval are assessed.
The fraction positive should be high in the relevant quiet wear examples and
low in confirmed removals. This compares the two window lengths; overlapping
windows are correlated, so their counts are not independent observations.


In [ ]:
# Fill with recording-local timestamps, for example:
# {"start": "...", "end": "...", "label": "wear"}
REVIEW_INTERVALS = pd.DataFrame([], columns=["start", "end", "label"])

review_metrics = pd.DataFrame()
if REVIEW_INTERVALS.empty:
    print("No independent labels supplied yet; no accuracy claim is made.")
else:
    labels = REVIEW_INTERVALS.copy()
    labels["start"] = pd.to_datetime(labels.start)
    labels["end"] = pd.to_datetime(labels.end)
    labels = labels.sort_values("start").reset_index(drop=True)
    if (not labels.label.isin(["wear", "nonwear"]).all()
            or labels[["start", "end"]].isna().any().any()
            or not labels.start.lt(labels.end).all()
            or (labels.start.iloc[1:].to_numpy() < labels.end.iloc[:-1].to_numpy()).any()):
        raise ValueError("Use non-overlapping intervals with valid start/end and wear/nonwear labels.")
    rows = []
    for seconds, analysis in results.items():
        for interval in labels.itertuples(index=False):
            windows = analysis.windows
            inside = ((windows.start >= interval.start) & (windows.end <= interval.end)
                      & windows.valid_axes.gt(0))
            values = windows.loc[inside, "respiration_present"]
            rows.append({"window_seconds": seconds, "label": interval.label,
                         "start": interval.start, "end": interval.end,
                         "assessed_windows": len(values),
                         "positive_windows": int(values.sum()),
                         "percent_positive": 100 * values.mean() if len(values) else np.nan})
    review_metrics = pd.DataFrame(rows)
    display(review_metrics)


## Optional: export a separate experiment

Leave `EXPORT_RESULTS = False` while exploring. Setting it to `True` creates a
new unique `respiration_...` folder under the private comparison output root.
It saves parameters/provenance, summary, per-axis window features, proposed
nonwear/respiration/reassignment bouts, independent labels and the selected figures.
The original comparison and pipeline outputs are preserved. Saved spectra are
optional because they can be large. Clear notebook outputs before committing.


In [ ]:
EXPORT_RESULTS = False
SAVE_SPECTRA = False

if EXPORT_RESULTS:
    from datetime import datetime, timezone
    import hashlib
    from uuid import uuid4

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    experiment_dir = OUTPUT_ROOT / f"respiration_{SUBJECT}_{VISIT}_{stamp}_{uuid4().hex[:8]}"
    experiment_dir.mkdir(parents=True, exist_ok=False)
    manifest = {
        "status": "running", "created_utc": stamp, "subject": SUBJECT, "visit": VISIT,
        "comparison_run": str(RUN_DIR), "recording": recording_info,
        "analysis_settings": {seconds: asdict(analysis.settings) for seconds, analysis in results.items()},
        "window_seconds": list(results),
        "overlap_by_window": {seconds: analysis.overlap for seconds, analysis in results.items()},
        "display": {"baseline_method": BASELINE_METHOD, "window_seconds": DISPLAY_WINDOW_SECONDS,
                    "start": PLOT_START, "end": PLOT_END, "inspected_window": selected_window,
                    "masked_daytime": MASKED_DAYTIME, "brpm_limits": BRPM_LIMITS,
                    "brpm_ylim": BRPM_YLIM},
        "baseline_methods": list(baselines), "sample_rate_hz": FS,
        "decision": "any positive axis; union of full positive windows; intersect existing nonwear only; then discard short nonwear bouts",
        "min_nonwear_bout_seconds": MIN_NONWEAR_BOUT_SECONDS,
        "core_sha256": hashlib.sha256((PROJECT_ROOT / "src" / "dare_wearables" / "lower_back" / "nonwear" / "respiration.py").read_bytes()).hexdigest(),
        "helper_sha256": hashlib.sha256((PROJECT_ROOT / "scripts" / "lower_back_respiration.py").read_bytes()).hexdigest(),
        "comparison_manifest_sha256": hashlib.sha256((RUN_DIR / "manifest.json").read_bytes()).hexdigest(),
        "lockfile_sha256": hashlib.sha256((PROJECT_ROOT / "uv.lock").read_bytes()).hexdigest(),
    }
    manifest_path = experiment_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, default=str) + "\n")
    try:
        respiration_summary.to_csv(experiment_dir / "summary.csv", index=False)
        REVIEW_INTERVALS.to_csv(experiment_dir / "review_intervals.csv", index=False)
        if not review_metrics.empty:
            review_metrics.to_csv(experiment_dir / "review_metrics.csv", index=False)
        for seconds, analysis in results.items():
            table = analysis.windows.copy()
            for method, baseline in baselines.items():
                table[f"{method}_nonwear_fraction"] = rp.nonwear_fraction(table, baseline, analysis.n_samples)
                for name in ("nonwear_bouts", "nonwear_bouts_before_duration_filter",
                             "short_nonwear_bouts", "rescued_bouts", "evidence_bouts"):
                    overrides[(seconds, method)][name].to_csv(
                        experiment_dir / f"{seconds}s_{method}_{name}.csv", index=False,
                    )
            table.to_csv(experiment_dir / f"{seconds}s_windows.csv", index=False)
            if SAVE_SPECTRA:
                np.savez_compressed(experiment_dir / f"{seconds}s_spectra.npz",
                                    frequency_hz=analysis.frequencies_hz, psd=analysis.psd)
        for name, figure in (("effect", effect_figure), ("overview", overview_figure),
                              ("window", window_figure), ("quality", quality_figure)):
            figure.savefig(experiment_dir / f"{name}.png", dpi=160, bbox_inches="tight")
    except Exception as error:
        manifest.update(status="failed", error=str(error))
        raise
    else:
        manifest["status"] = "completed"
    finally:
        manifest_path.write_text(json.dumps(manifest, indent=2, default=str) + "\n")
    print(f"Saved separate experiment: {experiment_dir}")


## What to establish before considering integration

- Inspect both window lengths in suspected sleep and confirmed removal intervals.
  The 60 s estimate offers finer frequency bins but spreads decisions across a
  longer interval; a 30 s window contains only three cycles at the lower band edge.
- Check the amplitude floor and peak concentration against genuine stationary
  sensor noise. A convincing-looking band-limited signal alone is insufficient.
- Check whether a stricter motion guard or persistence across multiple windows
  is needed. Neither is imposed silently: the current rule is the requested
  **one positive axis in one window**, and the motion guard is disabled initially.
- Reassignment is reversible and restricted to the original nonwear mask.
  Unassessed portions, invalid windows, gaps and incomplete tails are not rescued.
- This cannot establish a physiological breathing rate or quantify accuracy without
  independent labels. Any later production integration needs a separate decision.

Method references: [SciPy 1.17 periodogram documentation](https://docs.scipy.org/doc/scipy-1.17.0/reference/generated/scipy.signal.periodogram.html)
for PSD estimation and scaling; [inertial respiratory-rate feasibility study](https://pubmed.ncbi.nlm.nih.gov/33327531/)
for background (different sensor placement; it does not validate this lower-back screen).
